# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [ ]:
import os
import glob

from utils import prepare_helios_data

# Set up the project directory
project_dir = '/home/capheus/projects/tumba_fix'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

valid_rays_dir = os.path.join(project_dir, 'valid_rays')
if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir)

# Establish the object IDs for the leaves
leaf_object_ids = [0]
wood_object_ids = [1]

# Run the data preparation script
prepare_helios_data(
    input_dir=helios_dir, 
    output_dir=valid_rays_dir, 
    references_dir=references_dir, 
    leaf_object_ids=leaf_object_ids, 
    wood_object_ids=wood_object_ids, 
    debug=False
    )

Processing dask delayed functions...
[########################################] | 100% Completed | 102.32 ms
Processing leg 0...
[########################################] | 100% Completed | 2.91 ss
Processing leg 1...
[########################################] | 100% Completed | 3.44 ss
Helios data preparation complete.


Process 57c29fb5c02446009f6851f9e2282e43: Start 6878588 rays, 74957 voxels, in (74957) chunks.


## Compute Normals and Weights for Leaf Points

In [2]:
import os
import glob
from utils import add_normals_weights_to_valid_rays

valid_rays_dir = "/home/capheus/projects/tumba_fix/valid_rays"
valid_ray_parquets = glob.glob(os.path.join(valid_rays_dir, "*valid_rays.parquet"))

# Calculate normals and weights by loading valid rays
add_normals_weights_to_valid_rays(valid_ray_parquets, knn=6)

Adding normals and weights to 2 files...
[########################################] | 100% Completed | 2.22 ss


Estimating normals/weights per voxel: 100%|██████████| 102/102 [00:02<00:00, 38.21it/s]


Saved /home/capheus/projects/tumba_fix/valid_rays/leg_0_valid_rays.parquet
Saved /home/capheus/projects/tumba_fix/valid_rays/leg_1_valid_rays.parquet
Saved 2 valid rays files with normals and weights.


/home/capheus/PlantDensityAnalysis/utils.py:2870: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  


### Map missing values into old intersection files

In [6]:
import os
import glob
import pandas as pd

valid_rays_dir = '/home/capheus/projects/tumba_fix/valid_rays'
old_intersection_dir = valid_rays_dir

old_intersection_files = glob.glob(os.path.join(old_intersection_dir, '*intersection*.parquet'))

if not old_intersection_files:
    print("No old intersection files found.")

# Read each intersection file, and merge with same leg value valid_rays.parquet.
# Missing values of normal_x, normal_y, normal_z, weight, return_number, and number_of_returns from valid_rays
# added to intersection file, merging on ray_id, point_x, point_y, point_z
for intersection_file in old_intersection_files:
    intersection_parquet = pd.read_parquet(intersection_file)
    leg_value = intersection_parquet['leg_id'].unique()[0]

    valid_rays_file = os.path.join(valid_rays_dir, f'leg_{leg_value}_valid_rays.parquet')
    if not os.path.exists(valid_rays_file):
        print(f"Valid rays file for leg {leg_value} does not exist. Skipping intersection file {intersection_file}.")
        continue

    valid_rays_parquet = pd.read_parquet(valid_rays_file)
    valid_rays_parquet = valid_rays_parquet[['ray_id', 'point_x', 'point_y', 'point_z', 'echo_intensity', 'normal_x', 'normal_y', 'normal_z', 'point_weight', 'return_number', 'number_of_returns']]

    # Merge the dataframes on the specified columns
    merged_df = intersection_parquet.merge(
        valid_rays_parquet,
        on=['ray_id', 'point_x', 'point_y', 'point_z'],
        how='left',
        suffixes=('_old', '')
    )
    merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_old')]

    # Save the merged dataframe back to a parquet file
    output_file = os.path.join(valid_rays_dir, f'merged_{os.path.basename(intersection_file)}')
    merged_df.to_parquet(output_file)
    print(f"Merged data saved to {output_file}.")

    # Check Merge
    leaf_df = merged_df[merged_df['point_x'].isna() & merged_df['is_leaf'] == True]
    print(leaf_df.head())


Merged data saved to /home/capheus/projects/tumba_fix/valid_rays/merged_leg_0_voxel_2.0_intersections.parquet.
       voxel_size    voxel_id  leg_id   ray_id  t_entry_x  t_entry_y  \
24691         2.0  3603937916       0  3274170  32.067833  55.997677   
25400         2.0  3603937916       0  3266684  31.928560  55.997677   
26931         2.0  3603937916       0  3256372  31.738111  55.997677   
26934         2.0  3603937916       0  3256373  31.738392  55.997677   
26937         2.0  3603937916       0  3256374  31.738672  55.997677   

       t_entry_z   t_exit_x   t_exit_y   t_exit_z  ...  viewing_angle  \
24691  13.977233  31.983807  55.090420  14.462398  ...      61.965565   
25400  14.410441  31.919256  55.904377  14.462398  ...      61.008205   
26931  14.205741  31.686705  55.527973  14.462398  ...      61.490120   
26934  14.249230  31.695873  55.609142  14.462398  ...      61.392551   
26937  14.292799  31.704985  55.689800  14.462398  ...      61.294983   

       hit_ray  i

## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [ ]:
import os
from utils import voxel_ray_intersections

# Set up the project directory
project_dir = '/home/capheus/projects/tumba_fix'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')

# Run intersections
voxel_ray_intersections(
    valid_rays_dir=valid_rays_dir, 
    references_dir=references_dir
)

Queuing map_partitions tasks...
Number of partitions: 30, Number of rays: 6878588, Number of voxels: 3042, Rays per chunk: 235884, Voxels per chunk: 104, Number of chunks: 29
Queuing map_partitions tasks...
Number of partitions: 30, Number of rays: 6892044, Number of voxels: 3042, Rays per chunk: 236104, Voxels per chunk: 104, Number of chunks: 29
Processing leg 0...
[###############                         ] | 38% Completed | 227.98 s


## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

### Step 3.1 - Multi-return
Use the get_voxel_metrics_multireturn function to handle mutli return projects.

In [ ]:
import os
import glob
import utils
import pandas as pd

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = [2.0] # 'all' # [0.2, 0.5, 1.0, 2.0]

# Set the average leaf area
average_leaf_area = 0.02  # in m^2, adjust as needed

# Set up the project directory
project_dir = '/home/capheus/projects/200_leaf_60_JoshOutput'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')
# Set up the output directory
output_dir = os.path.join(project_dir, 'results')

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get the list of all voxel sizes
intersection_files = []
if legs == 'all' and voxel_sizes == 'all':
    intersection_files = glob.glob(os.path.join(valid_rays_dir, '*_intersections.parquet'))
elif legs == 'all' and isinstance(voxel_sizes, list):
    for voxel_size in voxel_sizes:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
elif isinstance(legs, list) and voxel_sizes == 'all':
    for leg in legs:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_*_intersections.parquet'))
else:
    for leg in legs:
        for voxel_size in voxel_sizes:
            intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_voxel_{voxel_size}_intersections.parquet'))

# Check if any intersection files were found
if intersection_files == []:
    print("No intersection files found. Please check the input parameters.")

# Split intersection files into separate lists for each voxel_size
voxel_size_files = {}
for file in intersection_files:
    # Extract the voxel size from the filename
    parts = file.split('_')
    voxel_size = float(parts[parts.index('voxel') + 1])
    
    # Add the file to the corresponding voxel size list
    if voxel_size not in voxel_size_files:
        voxel_size_files[voxel_size] = []
    voxel_size_files[voxel_size].append(file)

# Extract voxel information for each voxel size
for voxel_size, files in voxel_size_files.items():
    # Create a list of all legs in files
    legs = []
    for file in files:
        leg = os.path.basename(file)
        parts = leg.split('_')
        leg = int(parts[parts.index('leg') + 1])
        legs.append(leg)

    # Calculate the lambda_1 for average leaf area
    lambda_1 = utils.calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)
    print(f"Voxel size: {voxel_size}, Lambda_1: {lambda_1}")

    ### TEST
    lambda_1 = 0.00030701925

    # Calculate per voxel information from all files
    voxel_metrics_df = utils.get_voxel_metrics_multireturn(intersections_files=files, lambda_1=lambda_1, is_leaf_true=True)

    # Retrieve the reference file
    reference_file = glob.glob(os.path.join(references_dir, f'*voxel_size_{voxel_size}*'))[0]
    df_ref = pd.read_csv(reference_file)

    # CI_leaf_Corr, CI_lw_Corr
    # Ensure only numeric columns are included in the mean operation
    df_ref = df_ref.groupby('voxel_id').mean(numeric_only=True).reset_index()
    df_ref = df_ref.add_suffix('_ref')

    df_ref.rename(columns={
        'voxel_id_ref': 'voxel_id', 
        'voxel_cx_ref': 'voxel_cx',
        'voxel_cy_ref': 'voxel_cy',
        'voxel_cz_ref': 'voxel_cz',
        'LAD_ref_ref': 'LAD_ref', 
        'PAD_ref_ref': 'PAD_ref'
        }, inplace=True)

    # Merge to maintain voxel_id matching
    voxel_metrics_df = voxel_metrics_df.merge(df_ref, on='voxel_id', how='left')

    # Add lambda_1 to the dataframe
    voxel_metrics_df['lambda_1'] = lambda_1

    ### Add LAD calculations here if desired
    """Example, LAD_BL_TLS

    # Retrieve required variables
    I_leaf = voxel_metrics_df['I_leaf'].values
    mean_path_length = voxel_metrics_df['mean_path_length'].values  
    G_leaf = voxel_metrics_df['G_leaf'].values
    CI_leaf_ref = voxel_metrics_df['CI_leaf_corr_ref'].values

    LAD_BL_TLS = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length)
    LAD_BL_TLS_G = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf)
    LAD_BL_TLS_CI_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf, CI=CI_leaf_ref)
    """

    # Save outputs to csv
    project_name = os.path.basename(os.path.normpath(project_dir))
    legs.sort()
    leg_string = "_".join(map(str, legs))
    output_file = os.path.join(output_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
    if os.path.exists(output_file):
        os.remove(output_file)
    voxel_metrics_df.to_csv(output_file)

Voxel size: 2.0, Lambda_1: 0.0025
[########################################] | 100% Completed | 618.41 ms
